# 🌐 InvestAI — API REST v2 (FastAPI + MongoDB + ngrok)
**Versión 2.0** — expone todos los resultados de los 4 módulos de IA.

| Endpoint | Colección MongoDB |
|---|---|
| `GET /api/salud` | — |
| `GET /api/mercado/{ticker}` | `precios_ohlcv` |
| `GET /api/svc/{ticker}` | `predicciones` |
| `GET /api/rnn/{ticker}` | `predicciones_rnn` |
| `GET /api/rnn/{ticker}/{arq}` | `predicciones_rnn` |
| `GET /api/lstm/{ticker}` | `predicciones_lstm` |
| `GET /api/sentimiento/{ticker}` | `sentimiento_resumen` |
| `GET /api/sentimiento/{ticker}/noticias` | `sentimiento_noticias` |
| `GET /api/dashboard/{ticker}` | Todas (consolidado) |

> ⚠️ Secrets requeridos: `MONGO_URI` y `NGROK_AUTHTOKEN`.

In [ ]:
import requests

# Obtener la IP pública actual de la sesión de Colab
response = requests.get('https://api.ipify.org?format=json')
public_ip = response.json()['ip']
print(f"Tu dirección IP pública actual de Colab es: {public_ip}")

Tu dirección IP pública actual de Colab es: 34.81.114.169


## Sección 1 — Instalación

In [ ]:
!pip install fastapi uvicorn pyngrok "pymongo[srv]" nest-asyncio --quiet
print("✅ Dependencias instaladas.")

## Sección 2 — Importaciones

In [ ]:
import warnings, threading, time
warnings.filterwarnings('ignore')
from datetime import datetime
from typing import Optional

import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException, Path, Query
from fastapi.middleware.cors import CORSMiddleware

from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

from pyngrok import ngrok, conf
from pyngrok.exception import PyngrokNgrokError

from google.colab import userdata

print("✅ Importaciones OK.")

## Sección 3 — Configuración

In [ ]:
TICKERS_VALIDOS      = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
ARQUITECTURAS_VALIDAS = ['SimpleRNN', 'GRU', 'LSTM']

DB_NOMBRE     = 'investai'
COL_PRECIOS   = 'precios_ohlcv'
COL_SVC       = 'predicciones'
COL_RNN       = 'predicciones_rnn'
COL_LSTM      = 'predicciones_lstm'
COL_SENT_RES  = 'sentimiento_resumen'
COL_SENT_NOT  = 'sentimiento_noticias'
COL_METRICAS  = 'metricas_modelos'

EXCLUIR      = {'_id': 0, 'created_at': 0}
PUERTO_LOCAL = 8000

print("✅ Configuración OK.")

## Sección 4 — Conexión a MongoDB

In [ ]:
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("Secret MONGO_URI vacío.")
except Exception as e:
    raise RuntimeError(f"No se encontró MONGO_URI: {e}")

try:
    cliente_mongo = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10_000)
    cliente_mongo.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida.")
except ConnectionFailure as e:
    raise RuntimeError(f"Falla de conexión: {e}")

db           = cliente_mongo[DB_NOMBRE]
col_precios  = db[COL_PRECIOS]
col_svc      = db[COL_SVC]
col_rnn      = db[COL_RNN]
col_lstm     = db[COL_LSTM]
col_sent_res = db[COL_SENT_RES]
col_sent_not = db[COL_SENT_NOT]
col_metricas = db[COL_METRICAS]

cols_info = [
    (COL_PRECIOS, col_precios), (COL_SVC, col_svc),
    (COL_RNN, col_rnn), (COL_LSTM, col_lstm),
    (COL_SENT_RES, col_sent_res),
]
for nombre, col in cols_info:
    n = col.count_documents({})
    estado = '✅' if n > 0 else '⚠️ '
    print(f"   {estado} {nombre:<28}: {n:,} docs")

## Sección 5 — Configuración de ngrok

In [ ]:
try:
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
    if not NGROK_AUTHTOKEN:
        raise ValueError("Secret NGROK_AUTHTOKEN vacío.")
except Exception as e:
    raise RuntimeError(f"No se encontró NGROK_AUTHTOKEN: {e}")

# pyngrok 8.x: asignar authtoken mediante el objeto de configuración global
conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()   # Cerrar túneles previos si la celda se re-ejecuta

print("✅ Token ngrok configurado (pyngrok 8.x).")

## Sección 6 — Definición de la App FastAPI

In [ ]:
app = FastAPI(
    title="InvestAI API",
    description="API REST de solo lectura del Sistema InvestAI v2.0",
    version="2.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)

# CORS para cualquier origen: permite fetch() desde GitHub Pages u otro dominio
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

print("✅ App FastAPI v2 creada con CORS habilitado.")

In [ ]:
def serializar(obj):
    """
    Convierte recursivamente datetimes de MongoDB a strings ISO 8601.
    Necesario porque FastAPI no puede serializar datetime nativo de pymongo.
    """
    if isinstance(obj, dict):
        return {k: serializar(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [serializar(i) for i in obj]
    if isinstance(obj, datetime):
        return obj.isoformat()
    return obj


def not_found(coleccion, ticker, extra=''):
    return HTTPException(
        status_code=404,
        detail=(f"Sin datos en '{coleccion}' para '{ticker}'{extra}. "
                f"Ejecuta primero el notebook correspondiente.")
    )

print("✅ Helpers definidos.")

In [ ]:
@app.get('/api/salud', tags=['Sistema'],
         summary='Health check: estado del servidor y conteo de colecciones')
def api_salud():
    """
    Verifica conectividad con MongoDB y reporta documentos por colección.
    Usado por index.html para confirmar que la URL de ngrok es válida.
    """
    try:
        cliente_mongo.admin.command('ping')
        mongo_ok = True
    except Exception:
        mongo_ok = False

    def count(col):
        try:
            return col.count_documents({}) if mongo_ok else None
        except Exception:
            return None

    return {
        'status': 'ok' if mongo_ok else 'degradado',
        'servicio': 'InvestAI API', 'version': '2.0.0',
        'timestamp': datetime.now().isoformat(),
        'mongodb_conectado': mongo_ok,
        'tickers_disponibles': TICKERS_VALIDOS,
        'arquitecturas_rnn': ARQUITECTURAS_VALIDAS,
        'documentos': {
            'precios_ohlcv'       : count(col_precios),
            'predicciones_svc'    : count(col_svc),
            'predicciones_rnn'    : count(col_rnn),
            'predicciones_lstm'   : count(col_lstm),
            'sentimiento_resumen' : count(col_sent_res),
            'sentimiento_noticias': count(col_sent_not),
            'metricas_modelos'    : count(col_metricas),
        }
    }


@app.get('/', tags=['Sistema'], include_in_schema=False)
def raiz():
    return {
        'mensaje': 'InvestAI API v2.0 — Visita /docs para documentación interactiva',
        'endpoints': [
            '/api/salud', '/api/mercado/{ticker}', '/api/svc/{ticker}',
            '/api/rnn/{ticker}', '/api/rnn/{ticker}/{arq}',
            '/api/lstm/{ticker}', '/api/sentimiento/{ticker}',
            '/api/sentimiento/{ticker}/noticias', '/api/dashboard/{ticker}',
        ]
    }

print("✅ GET /api/salud")

In [ ]:
@app.get('/api/mercado/{ticker}', tags=['Mercado'],
         summary='Serie OHLCV + indicadores técnicos desde MongoDB')
def api_mercado(
    ticker: str = Path(..., description="Simbolo bursatil, ej. BVN"),
    ultimos: Optional[int] = Query(None, description="Limitar a los últimos N registros")
):
    """
    Devuelve la serie OHLCV + SMA-20/50, EMA-12/26, RSI-14 almacenada
    por el Módulo 1 (Ingesta). No descarga nada de Yahoo Finance.
    Usa el header 'ngrok-skip-browser-warning: true' en el frontend.
    """
    t = ticker.strip().upper()
    cursor = col_precios.find({'ticker': t}, EXCLUIR).sort('fecha', 1)
    serie  = [serializar(d) for d in cursor]
    if not serie:
        raise not_found(COL_PRECIOS, t)
    if ultimos and ultimos > 0:
        serie = serie[-ultimos:]
    ultimo = serie[-1]
    return {
        'ticker': t, 'nombre': ultimo.get('nombre', t),
        'moneda': ultimo.get('moneda', 'USD'),
        'total_registros': len(serie),
        'precio_actual': ultimo.get('cierre'),
        'fecha_actual': ultimo.get('fecha_str') or ultimo.get('fecha'),
        'serie': serie,
    }


@app.get('/api/svc/{ticker}', tags=['Modelos IA'],
         summary='Senal vigente del Clasificador SVC')
def api_svc(ticker: str = Path(..., description="Simbolo bursatil")):
    """Señal BUY/SELL del SVC con GridSearchCV + TimeSeriesSplit."""
    t   = ticker.strip().upper()
    doc = col_svc.find_one({'ticker': t, 'modelo': 'SVC'}, EXCLUIR)
    if not doc:
        raise not_found(COL_SVC, t)
    return serializar(doc)

print("✅ GET /api/mercado/{ticker}")
print("✅ GET /api/svc/{ticker}")

In [ ]:
@app.get('/api/rnn/{ticker}', tags=['Modelos IA'],
         summary='Senales de los 3 Clasificadores RNN para un ticker')
def api_rnn(ticker: str = Path(..., description="Simbolo bursatil")):
    """
    Devuelve las señales BUY/SELL de SimpleRNN, GRU y LSTM
    en una sola respuesta indexada por arquitectura.
    """
    t    = ticker.strip().upper()
    docs = [serializar(d) for d in
            col_rnn.find({'ticker': t}, EXCLUIR).sort('arquitectura', 1)]
    if not docs:
        raise not_found(COL_RNN, t)
    por_arq = {d['arquitectura']: d for d in docs}
    return {
        'ticker': t, 'modelo': 'RNN',
        'arquitecturas': ARQUITECTURAS_VALIDAS,
        'por_arquitectura': por_arq,
        'senales': {a: por_arq.get(a, {}).get('senal', '?')
                    for a in ARQUITECTURAS_VALIDAS},
    }


@app.get('/api/rnn/{ticker}/{arquitectura}', tags=['Modelos IA'],
         summary='Senal de una arquitectura RNN especifica')
def api_rnn_arq(
    ticker: str       = Path(..., description="Simbolo bursatil"),
    arquitectura: str = Path(..., description="SimpleRNN, GRU o LSTM")
):
    t   = ticker.strip().upper()
    arq = next((a for a in ARQUITECTURAS_VALIDAS
                if a.lower() == arquitectura.lower()), arquitectura)
    doc = col_rnn.find_one({'ticker': t, 'arquitectura': arq}, EXCLUIR)
    if not doc:
        raise not_found(COL_RNN, t, f" / arquitectura '{arq}'")
    return serializar(doc)

print("✅ GET /api/rnn/{ticker}")
print("✅ GET /api/rnn/{ticker}/{arquitectura}")

In [ ]:
@app.get('/api/lstm/{ticker}', tags=['Modelos IA'],
         summary='Predicciones de precios futuros del Regresor LSTM')
def api_lstm(
    ticker: str = Path(..., description="Simbolo bursatil"),
    horizonte: Optional[int] = Query(None, description="Filtrar horizonte: 7, 14, 30 o 60")
):
    """
    Predicciones LSTM en USD con bandas de confianza 95% (+-1.96xRMSE).
    Parámetro ?horizonte=N (7/14/30/60) para recibir solo un horizonte.
    """
    t = ticker.strip().upper()
    proj = dict(EXCLUIR)
    if horizonte:
        proj['historico_test'] = 0   # Omitir historial si solo se pide un horizonte
    doc = col_lstm.find_one({'ticker': t}, proj)
    if not doc:
        raise not_found(COL_LSTM, t)
    doc = serializar(doc)
    if horizonte:
        h_str = str(horizonte)
        pf = doc.get('predicciones_futuras', {})
        if h_str not in pf:
            raise HTTPException(404,
                detail=f"Horizonte '{horizonte}' no disponible. Disponibles: {list(pf.keys())}")
        return {
            'ticker': t, 'modelo': 'LSTM_Regressor', 'horizonte': horizonte,
            'metricas': doc.get('metricas'), 'prediccion': pf[h_str],
        }
    return doc


@app.get('/api/sentimiento/{ticker}', tags=['Sentimiento NLP'],
         summary='Resumen de sentimiento VADER para un ticker')
def api_sentimiento(ticker: str = Path(..., description="Simbolo bursatil")):
    """Score compound promedio, distribución y últimas 5 noticias con scores VADER."""
    t   = ticker.strip().upper()
    doc = col_sent_res.find_one({'ticker': t}, EXCLUIR)
    if not doc:
        raise not_found(COL_SENT_RES, t)
    return serializar(doc)


@app.get('/api/sentimiento/{ticker}/noticias', tags=['Sentimiento NLP'],
         summary='Noticias individuales con scores VADER')
def api_sentimiento_noticias(
    ticker: str = Path(..., description="Simbolo bursatil"),
    limite: Optional[int] = Query(20, description="Max noticias (default 20, max 100)")
):
    t   = ticker.strip().upper()
    lim = min(max(1, limite or 20), 100)
    docs = [serializar(d) for d in
            col_sent_not.find({'ticker': t}, EXCLUIR)
            .sort('fecha_publicacion', -1).limit(lim)]
    if not docs:
        raise not_found(COL_SENT_NOT, t)
    return {'ticker': t, 'total': len(docs), 'limite': lim, 'noticias': docs}

print("✅ GET /api/lstm/{ticker}")
print("✅ GET /api/sentimiento/{ticker}")
print("✅ GET /api/sentimiento/{ticker}/noticias")

In [ ]:
@app.get('/api/dashboard/{ticker}', tags=['Dashboard'],
         summary='Todos los modelos consolidados en una peticion')
def api_dashboard(ticker: str = Path(..., description="Simbolo bursatil")):
    """
    Endpoint de conveniencia: consolida mercado + SVC + RNN + LSTM + Sentimiento
    en una sola respuesta. Evita que el frontend haga 5+ peticiones separadas.
    """
    t = ticker.strip().upper()

    doc_mercado = col_precios.find_one({'ticker': t}, EXCLUIR, sort=[('fecha', -1)])
    doc_svc     = col_svc.find_one({'ticker': t, 'modelo': 'SVC'}, EXCLUIR)
    docs_rnn    = list(col_rnn.find({'ticker': t}, EXCLUIR).sort('arquitectura', 1))
    doc_lstm    = col_lstm.find_one({'ticker': t}, {**EXCLUIR, 'historico_test': 0})
    doc_sent    = col_sent_res.find_one({'ticker': t}, EXCLUIR)

    rnn_por_arq = {d['arquitectura']: serializar(d) for d in docs_rnn}

    return {
        'ticker': t,
        'actualizado': datetime.now().isoformat(),
        'mercado':     serializar(doc_mercado) if doc_mercado else None,
        'svc':         serializar(doc_svc)     if doc_svc     else None,
        'rnn':         rnn_por_arq             if rnn_por_arq else None,
        'lstm':        serializar(doc_lstm)    if doc_lstm    else None,
        'sentimiento': serializar(doc_sent)    if doc_sent    else None,
    }

print("✅ GET /api/dashboard/{ticker}")
print()
print("Endpoints registrados:")
for r in app.routes:
    if hasattr(r, 'methods'):
        print(f"  {list(r.methods)[0]:<6} {r.path}")

## Sección 7 — Despliegue: ngrok + uvicorn
Esta celda abre el túnel ngrok y lanza uvicorn en un **hilo separado**.
Quedará bloqueada mientras el servidor esté activo.
Presiona **⏹ Stop** para detenerlo.

In [ ]:
nest_asyncio.apply()

url_publica   = None
server_thread = None

def run_uvicorn(app_inst, host, port):
    uvicorn.run(app_inst, host=host, port=port, log_level='info')

try:
    tunel       = ngrok.connect(PUERTO_LOCAL)
    url_publica = tunel.public_url

    print("=" * 70)
    print("  InvestAI API v2.0 — Servidor desplegado")
    print("=" * 70)
    print(f"  URL publica   : {url_publica}")
    print(f"  Swagger UI    : {url_publica}/docs")
    print(f"  Health check  : {url_publica}/api/salud")
    print(f"  Dashboard BVN : {url_publica}/api/dashboard/BVN")
    print("=" * 70)
    print()
    print("✅ Copia la URL publica en index.html para conectar el frontend.")
    print("   Presiona Stop para apagar el servidor.")

    server_thread = threading.Thread(
        target=run_uvicorn,
        args=(app, '0.0.0.0', PUERTO_LOCAL),
        daemon=True
    )
    server_thread.start()

    while server_thread.is_alive():
        time.sleep(1)

except PyngrokNgrokError as e:
    print(f"Error ngrok: {e}")
except KeyboardInterrupt:
    print("Servidor detenido manualmente.")
except Exception as e:
    import traceback
    traceback.print_exc()
finally:
    if url_publica:
        ngrok.disconnect(url_publica)
        print("Tunel ngrok cerrado.")

## Sección 8 — Guía de Integración Frontend

Todos los `fetch()` del frontend deben incluir el header anti-advertencia de ngrok:

```javascript
const BASE = sessionStorage.getItem('INVESTAI_API_URL');

const res = await fetch(`${BASE}/api/dashboard/BVN`, {
  headers: {
    'Accept': 'application/json',
    'ngrok-skip-browser-warning': 'true'
  }
});
const data = await res.json();
```

| HTML | Endpoint |
|---|---|
| `index.html` | `/api/salud` |
| `modulo_mercado.html` | `/api/mercado/{ticker}?ultimos=120` |
| `modulo_svc.html` | `/api/svc/{ticker}` |
| `modulo_rnn.html` | `/api/rnn/{ticker}` |
| `modulo_lstm.html` | `/api/lstm/{ticker}?horizonte=30` |
| `modulo_sentimiento.html` | `/api/sentimiento/{ticker}` |
| `dashboard_completo.html` | `/api/dashboard/{ticker}` |